In [1]:
from dataclasses import dataclass
from enum import Enum, auto
import numpy as np

In [2]:
class Op(Enum):
    LE = auto()
    LEQ = auto()
    GE = auto()
    GEQ = auto()
    EQ = auto()

    def __repr__(self):
        match self:
            case Op.LE:
                return "<"
            case Op.LEQ:
                return "<="
            case Op.GE:
                return ">"
            case Op.GEQ:
                return ">="
            case Op.EQ:
                return "="

In [3]:
@dataclass(frozen=True, slots=True)
class Expression:
    terms: dict

    def __repr__(self):
        return " + ".join(
            f"{coefficient}{var.symbol}"
            for var, coefficient in self.terms.items()
            if coefficient != 0
        ) or "0"

    @staticmethod
    def from_value(value):
        if isinstance(value, Expression):
            return value

        if isinstance(value, Var):
            return Expression({value: 1})

        if isinstance(value, (int, float)):
            return Expression({})

        raise TypeError(f"Cannot convert {type(value)} to Expression")
    
    def __add__(self, other):
        other = self.from_value(other)
        terms = self.terms.copy()
        
        for var, coefficient in other.terms.items():
            terms[var] = terms.get(var, 0) + coefficient
            
        return Expression(terms)

    def __sub__(self, other):
        other = self.from_value(other)
        terms = self.terms.copy()
        
        for var, coefficient in other.terms.items():
            terms[var] = terms.get(var, 0) - coefficient
            
        return Expression(terms)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return Expression({
                var: coefficient * other
                for var, coefficient in self.terms.items()
            })

        return NotImplemented

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        other = self.from_value(other)
        if isinstance(other, (int, float)):
            return Expression({
                var: coefficient / other
                for var, coefficient in self.terms.items()
            })

        return NotImplemented

    def __floordiv__(self, other):
        other = self.from_value(other)
        if isinstance(other, (int, float)):
            return Expression({
                var: coefficient // other
                for var, coefficient in self.terms.items()
            })

        return NotImplemented

    def __le__(self, other):
        return Equation(self, Op.LEQ, other)

    def __ge__(self, other):
        return Equation(self, Op.GEQ, other)
        
@dataclass(frozen=True, slots=True)
class Equation:
    lhs: Expression
    op: Op
    rhs: float

    def __repr__(self):
        return f"{self.lhs} {self.op.__repr__()} {self.rhs}"
        
@dataclass(frozen=True, slots=True)
class Var:
    symbol: str

    def __mul__(self, other):
        return Expression({self: 1}) * other

    def __rmul__(self, other):
        return self * other

    def __add__(self, other):
        return Expression({self: 1}) + other

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return Expression({self: 1}) - other

    def __rsub__(self, other):
        return -1 * self + other
        
    def __rmul__(self, coefficient):
        if isinstance(coefficient, (int, float)):
            return Expression({self: coefficient})
        return NotImplemented

    def __repr__(self):
        return self.symbol

In [4]:
x = Var("x")
y = Var("y")

In [5]:
objective = 30*x + 40*y
objective

30x + 40y

In [6]:
constraints = [
    2*x   +   y  <= 10,
      x   +   y  <=  7,
      x   + 2*y  <= 12
]
constraints

[2x + 1y <= 10, 1x + 1y <= 7, 1x + 2y <= 12]

In [7]:
obj_equality = objective*-1 <= 0
obj_equality

-30x + -40y <= 0

In [8]:
def equation_to_row(equation, variables, slack=None):
    terms = equation.lhs.terms
    row = [
        terms.get(var, 0) + (var == slack)
        for var in variables
    ]
    return row + [equation.rhs]


def to_tableau(objective, constraints):
    objective = objective * -1 <= 0

    variables = []

    for equation in constraints + [objective]:
        for var in equation.lhs.terms:
            if var not in variables:
                variables.append(var)

    slacks = [
        Var(f"s{i}")
        for i in range(1, len(constraints) + 1)
    ]

    variables += slacks

    rows = [
        equation_to_row(constraint, variables, slack)
        for constraint, slack in zip(constraints, slacks)
    ]

    rows.append(equation_to_row(objective, variables))

    return np.array(rows, dtype=float), variables

In [51]:
def pivot(input_tbl, debug=False):
    def log(v):
        if debug:
            print(v)
    tbl = input_tbl.copy()
    col_pivot = tbl[-1, :-1].argmin()
    log(tbl[-1, :-1])
    if tbl[-1, :-1][col_pivot] >= 0:
        log("No more pivots possible")
        return tbl, True
    log(f"Pivoting on index {col_pivot} ({tbl[-1, :-1][col_pivot]}) as its the lowest value")
    
    pivot_candidates = tbl[:-1, -1, None] / tbl[:-1, col_pivot, None]
    log(pivot_candidates)
    masked_pivots = np.where(pivot_candidates > 0, pivot_candidates, np.inf)
    row_pivot = masked_pivots.argmin()
    if np.all(np.isinf(masked_pivots)):
        raise Exception("Problem is unbounded")
    log(f"row pivot on {row_pivot}")
    
    log(f"Pivots: Row: {row_pivot}, Col: {col_pivot}")

    # Divide row by coefficient in pivot_col to reduce that coefficient to 1
    reduction_factor = tbl[row_pivot][col_pivot]
    tbl[row_pivot] = tbl[row_pivot] / reduction_factor

    log(tbl)
    # Now zero out coefficient in non pivot_row rows
    for idx, row in enumerate(tbl):
        if idx == row_pivot:
            continue
        coeff = row[col_pivot]
        row_to_add = (-1 * coeff) * tbl[row_pivot]
        tbl[idx] = row_to_add + row
    log(tbl)
    return tbl, False

In [52]:
def solve(objective, constraints):
    tbl, variables = to_tableau(objective, constraints)
    final_tableau = None
    while True:
        tbl, finished = pivot(tbl)
        if finished:
            final_tableau = tbl
            break
    assert final_tableau is not None
    A = final_tableau[:-1, :-1]
    basic_cols = np.where(
        ((A == 1).sum(axis=0) == 1) & # Sum is 1
        ((A == 0).sum(axis=0) == A.shape[0] - 1) # All other rows are 0's
    )[0]
    basic_rows = np.argmax(A[:, basic_cols] == 1, axis=0)
    basic_values = final_tableau[basic_rows, -1]
    solution = {
        variables[col]: value
        for col, value in zip(basic_cols, basic_values)
    }
    return solution

In [53]:
solve(objective, constraints)

{x: np.float64(2.0), y: np.float64(5.0), s1: np.float64(1.0)}